In [1]:
import pandas as pd

df = pd.read_csv("/content/Ecommerce_Behavior.csv")

print(df.head())
print(df.shape)

  Customer_ID Product_ID  Product_Name        Category Interaction_Type  \
0        C001       P024  Cookware Set  Home & Kitchen             View   
1        C001       P002    Smartphone     Electronics         Purchase   
2        C001       P025  Water Bottle  Home & Kitchen             View   
3        C001       P025  Water Bottle  Home & Kitchen         Purchase   
4        C001       P019        Wallet         Fashion             View   

   Rating            Timestamp  
0       0  2026-01-19 01:27:00  
1       1  2026-03-04 01:30:00  
2       0  2026-03-18 14:43:00  
3       4  2026-03-29 08:09:00  
4       0  2026-05-14 00:24:00  
(1000, 7)


In [2]:
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())

df = df.drop_duplicates()

print("After preprocessing:", df.shape)

Customer_ID         0
Product_ID          0
Product_Name        0
Category            0
Interaction_Type    0
Rating              0
Timestamp           0
dtype: int64
Duplicates: 0
After preprocessing: (1000, 7)


In [3]:
print("Popular Products:")
print(df["Product_Name"].value_counts().head(10))

print("\nPopular Categories:")
print(df["Category"].value_counts())

Popular Products:
Product_Name
Sunglasses       47
Backpack         40
Tablet           40
Jeans            39
Water Bottle     39
Air Fryer        39
Headphones       38
Mixer Grinder    38
Shirt            36
Table Lamp       36
Name: count, dtype: int64

Popular Categories:
Category
Fashion           343
Home & Kitchen    337
Electronics       320
Name: count, dtype: int64


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

data = df[df["Interaction_Type"] == "Purchase"]

matrix = pd.crosstab(data["Customer_ID"], data["Product_ID"])

similarity = cosine_similarity(matrix)

similarity_df = pd.DataFrame(
    similarity,
    index=matrix.index,
    columns=matrix.index
)

print(similarity_df.head())

Customer_ID      C001      C002      C003      C005      C006      C007  C008  \
Customer_ID                                                                     
C001         1.000000  0.471405  0.258199  0.288675  0.235702  0.333333   0.0   
C002         0.471405  1.000000  0.182574  0.408248  0.333333  0.000000   0.0   
C003         0.258199  0.182574  1.000000  0.000000  0.182574  0.000000   0.0   
C005         0.288675  0.408248  0.000000  1.000000  0.612372  0.288675   0.0   
C006         0.235702  0.333333  0.182574  0.612372  1.000000  0.000000   0.0   

Customer_ID  C009      C010      C012  ...      C089  C090  C092      C093  \
Customer_ID                            ...                                   
C001          0.0  0.000000  0.000000  ...  0.408248   0.0   0.0  0.288675   
C002          0.0  0.182574  0.000000  ...  0.288675   0.0   0.0  0.204124   
C003          0.0  0.000000  0.000000  ...  0.000000   0.0   0.0  0.223607   
C005          0.0  0.223607  0.250000  ...

In [5]:
customer = "C001"

similar_customers = similarity_df[customer].sort_values(ascending=False)[1:4]

print("Similar Customers:")
print(similar_customers)

products = matrix.loc[similar_customers.index].sum()
products = products.sort_values(ascending=False)

bought = matrix.loc[customer]
recommendations = products[bought == 0].head(5)

print("\nRecommended Product IDs:")
print(recommendations)

Similar Customers:
Customer_ID
C019    0.577350
C072    0.577350
C082    0.522233
Name: C001, dtype: float64

Recommended Product IDs:
Product_ID
P009    2
P026    1
P010    1
P003    0
P006    0
dtype: int64


In [6]:
recommended_ids = recommendations.index

result = df[df["Product_ID"].isin(recommended_ids)][
    ["Product_ID", "Product_Name", "Category"]
].drop_duplicates()

print("Recommended Products:")
print(result)

Recommended Products:
    Product_ID       Product_Name        Category
7         P010         Power Bank     Electronics
28        P006  Bluetooth Speaker     Electronics
33        P003         Headphones     Electronics
71        P009            Monitor     Electronics
101       P026           Bedsheet  Home & Kitchen
